In [ ]:
import pandas as pd
import random
import numpy as np
from scipy.stats import ks_2samp
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold



c:\Users\Toranj\Desktop\AIM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# all the test cases for checking data leakage 
# and ensuringto avoid any leakage frome time stamps of clip sampling 



LOG = "nexar_data/processed_frames_matched2/processing_log.csv"

META = "nexar_data/metadata.csv"


def part_a():
    log = pd.read_csv(LOG)
    log = log[log.status.isin(["ok", "ok_existing"])]
    print(log.groupby("label").size())  
    meta = pd.read_csv(META)[["video_id", "light_conditions", "weather", "scene"]]
    d = log.merge(meta, on="video_id")
    d["time_after_clip"] = d["video_duration"] - d["end_time"] 
    pos, neg = d[d.label == 1], d[d.label == 0]
    print(d.groupby("label")["time_after_clip"].describe())
    print("== Timing distributions (positive vs negative) ==")
    for col in ["start_time", "video_duration"]:
        ks = ks_2samp(pos[col], neg[col])
        print(f"{col}: pos mean={pos[col].mean():.2f}  neg mean={neg[col].mean():.2f}  "
              f"KS={ks.statistic:.3f} p={ks.pvalue:.3g}")

    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    sets = {
    "start_time only":     ["start_time"],
    "video_duration only": ["video_duration"],
    "time_after_clip only": ["time_after_clip"],              
    "timing only":         ["start_time", "video_duration"],
    "metadata only":       ["light_conditions", "weather", "scene"],
    "timing + metadata":   ["start_time", "video_duration", "light_conditions", "weather", "scene"],
    }
    print("\n== Non-visual baselines (ROC-AUC, ~0.50 means no shortcut) ==")
    for name, cols in sets.items():
        X = pd.get_dummies(d[cols], dtype=float)
        auc = cross_val_score(GradientBoostingClassifier(random_state=0),
                              X, d.label, cv=cv, scoring="roc_auc")
        print(f"{name}: AUC = {auc.mean():.3f} +/- {auc.std():.3f}")


def sample_matched_start(pos_starts, duration, clip_seconds, rng):
    """Draw a negative start time from the positives' empirical start-time
    distribution, restricted to values that fit inside this video."""
    pos_starts = np.asarray(pos_starts, dtype=float)
    ok = pos_starts[pos_starts + clip_seconds <= duration]
    if len(ok) == 0:
        return rng.uniform(0.0, duration - clip_seconds)
    return float(rng.choice(ok))


import torch
from sklearn.metrics import f1_score


def static_frame(x): 
    return x[:, :, :1].expand(-1, -1, x.shape[2], -1, -1).contiguous()


def borders_only(x):  
    m = torch.ones_like(x)
    h, w = x.shape[-2:]
    m[..., h // 4: 3 * h // 4, w // 4: 3 * w // 4] = 0
    return x * m


def center_only(x):  
    m = torch.zeros_like(x)
    h, w = x.shape[-2:]
    m[..., h // 4: 3 * h // 4, w // 4: 3 * w // 4] = 1
    return x * m


def shuffle_time(x):  # destroy temporal order
    return x[:, :, torch.randperm(x.shape[2])]


@torch.no_grad()
def macro_f1(model, loader, device, transform=lambda x: x):
    model.eval()
    preds, targets = [], []
    for videos, labels in loader:
        out = model(transform(videos).to(device))
        preds += out.argmax(1).cpu().tolist()
        targets += labels.tolist()
    return f1_score(targets, preds, average="macro", zero_division=0)


def part_c(model, val_loader, device):
    variants = {
        "normal": lambda x: x,
        "static frame": static_frame,
        "shuffled time": shuffle_time,
        "borders only": borders_only,
        "center only": center_only,
    }
    for name, fn in variants.items():
        print(f"{name:14s} val macro-F1 = {macro_f1(model, val_loader, device, fn):.3f}")


if __name__ == "__main__":
    part_a()


label
0    616
1    748
dtype: int64
       count       mean       std       min        25%        50%        75%  \
label                                                                          
0      616.0  20.076717  4.297856  4.218855  20.462373  21.214597  21.823391   
1      748.0  20.257115  3.558239  4.200000  20.566917  21.199667  21.777753   

             max  
label             
0      30.335977  
1      30.544883  
== Timing distributions (positive vs negative) ==
start_time: pos mean=14.14  neg mean=14.07  KS=0.034 p=0.806
video_duration: pos mean=38.40  neg mean=38.15  KS=0.030 p=0.914

== Non-visual baselines (ROC-AUC, ~0.50 means no shortcut) ==
start_time only: AUC = 0.655 +/- 0.021
video_duration only: AUC = 0.491 +/- 0.032
time_after_clip only: AUC = 0.636 +/- 0.019
timing only: AUC = 0.654 +/- 0.018
metadata only: AUC = 0.640 +/- 0.027
timing + metadata: AUC = 0.695 +/- 0.019


In [ ]:
proc = pd.read_csv("nexar_data/processed_frames_matched2/processing_log.csv")
m = proc[proc.status.isin(["ok","ok_existing"])].copy()
P = m[m.label==1]; N = m[m.label==0]

# comparing  time_after_clip histograms
print("pos after quantiles:", np.quantile(P.time_after_clip, [0,.25,.5,.75,1]).round(2))
print("neg after quantiles:", np.quantile(N.time_after_clip, [0,.25,.5,.75,1]).round(2))

pos after quantiles: [ 4.2  20.57 21.2  21.78 30.54]
neg after quantiles: [ 4.22 20.46 21.21 21.82 30.34]


In [ ]:
# checking out duplicates 

print("dup video_ids in m:", m.video_id.duplicated().sum())
print("overlap pos/neg video_ids:", len(set(m[m.label==1].video_id) & set(m[m.label==0].video_id)))

dup video_ids in m: 0
overlap pos/neg video_ids: 0


In [ ]:
m = proc[proc.status.isin(["ok","ok_existing"])].dropna(
        subset=["start_time","time_after_clip","video_duration"]).copy()
P, N = m[m.label==1], m[m.label==0]

print("n pos/neg:", len(P), len(N))
print("dup video_ids:", m.video_id.duplicated().sum())
print("pos∩neg video_ids:", len(set(P.video_id) & set(N.video_id)))

print("start_time values fed to AUC:")
print("  pos:", P.start_time.describe()[["mean","std","min","max"]].round(3).to_dict())
print("  neg:", N.start_time.describe()[["mean","std","min","max"]].round(3).to_dict())
print("  KS:", ks_2samp(P.start_time, N.start_time))



X = m[["start_time"]].astype(float).values
y = m.label.values
skf = StratifiedKFold(5, shuffle=True, random_state=42)
for i,(tr,te) in enumerate(skf.split(X,y)):
    sc = StandardScaler().fit(X[tr])
    lr = LogisticRegression(max_iter=1000).fit(sc.transform(X[tr]), y[tr])
    print(f"  fold {i}: AUC={roc_auc_score(y[te], lr.predict_proba(sc.transform(X[te]))[:,1]):.3f}"
          f"  | n_tr={len(tr)} n_te={len(te)}")

n pos/neg: 748 616
dup video_ids: 52
pos∩neg video_ids: 0
start_time values fed to AUC:
  pos: {'mean': 14.143, 'std': 3.477, 'min': 0.833, 'max': 51.8}
  neg: {'mean': 14.073, 'std': 3.296, 'min': 0.85, 'max': 31.247}
  KS: KstestResult(statistic=np.float64(0.03428189457601222), pvalue=np.float64(0.8060380091579671), statistic_location=np.float64(5.696999999999999), statistic_sign=np.int8(1))
  fold 0: AUC=0.489  | n_tr=1091 n_te=273
  fold 1: AUC=0.523  | n_tr=1091 n_te=273
  fold 2: AUC=0.537  | n_tr=1091 n_te=273
  fold 3: AUC=0.454  | n_tr=1091 n_te=273
  fold 4: AUC=0.466  | n_tr=1092 n_te=272


In [ ]:


proc = pd.read_csv("nexar_data/processed_frames_matched2/processing_log.csv")
m = proc[proc.status.isin(["ok", "ok_existing"])].copy()
m = m.dropna(subset=["start_time", "time_after_clip", "video_duration"])

m_dedup = (m.sort_values(["video_id", "label"])
             .drop_duplicates(subset="video_id", keep="first")
             .reset_index(drop=True))
print("rows before dedup:", len(m), "| after:", len(m_dedup),
      "| dups removed:", len(m) - len(m_dedup))
print("class balance after dedup:\n", m_dedup.label.value_counts())

P = m_dedup[m_dedup.label == 1]
N = m_dedup[m_dedup.label == 0]
n_min = min(len(P), len(N))
rng = random.Random(42)
P_bal = P.sample(n=n_min, random_state=42) if len(P) > n_min else P
N_bal = N.sample(n=n_min, random_state=42) if len(N) > n_min else N
m_bal = pd.concat([P_bal, N_bal]).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"balanced: {len(m_bal)} rows | pos={int((m_bal.label==1).sum())} neg={int((m_bal.label==0).sum())}")

y = m_bal.label.values
groups = m_bal.video_id.values
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

def group_auc(feats):
    X = m_bal[feats].astype(float).values
    aucs = []
    for tr, te in skf.split(X, y, groups):
        sc = StandardScaler().fit(X[tr])
        lr = LogisticRegression(max_iter=1000).fit(sc.transform(X[tr]), y[tr])
        if len(np.unique(y[te])) == 2:
            aucs.append(roc_auc_score(y[te], lr.predict_proba(sc.transform(X[te]))[:, 1]))
    return float(np.mean(aucs)), float(np.std(aucs))

print("\n=== Group-aware non-visual baselines (target <= 0.56) ===")
for name, feats in [
    ("start_time only",      ["start_time"]),
    ("time_after_clip only", ["time_after_clip"]),
    ("video_duration only",  ["video_duration"]),
    ("timing only",          ["start_time", "time_after_clip"]),
    ("timing + duration",    ["start_time", "time_after_clip", "video_duration"]),
]:
    a, s = group_auc(feats)
    flag = "✅" if a <= 0.56 else "⚠️"
    print(f"  {flag} AUC {name:22s}: {a:.3f} +/- {s:.3f}")

P2, N2 = m_bal[m_bal.label == 1], m_bal[m_bal.label == 0]
for col in ["start_time", "time_after_clip", "video_duration"]:
    ks = ks_2samp(P2[col], N2[col])
    print(f"KS {col:16s}: stat={ks.statistic:.3f} p={ks.pvalue:.3f}")

m_bal.to_csv("nexar_data/processed_frames_matched2/clean_balanced_log.csv", index=False)
print("\nsaved: clean_balanced_log.csv  (use THIS for the training split)")

rows before dedup: 1364 | after: 1312 | dups removed: 52
class balance after dedup:
 label
1    748
0    564
Name: count, dtype: int64
balanced: 1128 rows | pos=564 neg=564

=== Group-aware non-visual baselines (target <= 0.56) ===
  ✅ AUC start_time only       : 0.493 +/- 0.012
  ✅ AUC time_after_clip only  : 0.483 +/- 0.013
  ✅ AUC video_duration only   : 0.486 +/- 0.026
  ✅ AUC timing only           : 0.486 +/- 0.020
  ✅ AUC timing + duration     : 0.486 +/- 0.020
KS start_time      : stat=0.032 p=0.937
KS time_after_clip : stat=0.066 p=0.177
KS video_duration  : stat=0.032 p=0.937

saved: clean_balanced_log.csv  (use THIS for the training split)
